## 🚀 Creating a Data Pipeline

### What is a Pipeline in ADF?
In Azure Data Factory, a **Pipeline** is a logical grouping of activities that together perform a specific task. It acts as an orchestrator, defining the sequence of operations you want to execute (like moving data, transforming it, or triggering other processes).

### ➕ Steps to Create the Pipeline

**1. Navigate to the Author Tab**
* On the left-hand menu of Azure Data Factory Studio, click on the **Author** icon (it looks like a pencil). This is the workspace where all development of pipelines, datasets, and data flows happens.

**2. Create a New Pipeline**
* In the Factory Resources pane on the left, click the **+** (Plus) button at the top.
* Select **Pipeline** from the dropdown menu, and then click **Pipeline** again.
* *Alternative method:* Hover over the **Pipelines** category, click the three dots (**...**), and select **New pipeline**.

**3. Configure Pipeline Properties**
* Once the blank pipeline canvas opens, look at the **Properties** pane on the right side of the screen.
* Under the **General** section, locate the **Name** field.
* Change the default name (e.g., *pipeline1*) to: `onprem_ingestion`
* *(Optional)* Add a description, such as "Pipeline to ingest local on-premises files into Azure Data Lake Storage Gen2."

✅ **Result**: You now have a ready-to-use canvas for your `onprem_ingestion` pipeline! The next step is to drag and drop activities (like the **Copy Data** activity) onto this canvas to instruct ADF on how to move your files.

## 📋 Configuring the Copy Data Activity (Updated for CSV Sink)

### What is the Copy Data Activity?
The **Copy Data** activity is one of the most fundamental and widely used activities in Azure Data Factory. It securely reads data from a source data store and writes it to a destination (sink) data store. In this step, we are using it to migrate our local on-premises CSV file into our cloud Data Lake as a new CSV file.

### ➕ Steps to Configure the Copy Activity

**1. Add and Name the Activity**
* From the **Activities** pane on the left, expand the **Move and transform** section.
* Drag and drop the **Copy data** activity onto the pipeline canvas.
* In the **General** tab at the bottom, set the **Name** to: `MigrateOnPremFile`

**2. Configure the Source (Where data comes from)**
* Go to the **Source** tab and click **+ New** to create a source dataset.
* Search for and select **File System**.
* Choose **CSV (DelimitedText)** as the format.
* **Dataset Name**: `ds_prem` (Dataset for on-premises file).
* **Linked service**: Select the File System linked service you created earlier.
* **Import schema**: Select **None** (we are just moving the file as-is for now).

**3. Configure the Sink (Where data is going)**
* Go to the **Sink** tab and click **+ New** to create a destination dataset.
* Search for and select **Azure Data Lake Storage Gen2**.
* Choose **CSV (DelimitedText)** as the format. *(Note: By using CSV instead of Parquet, you completely bypass the Java/JreNotFound error you saw earlier, as CSV does not require a Java Runtime Environment to process!)*
* **Dataset Name**: `ds_onpremsink_csv` 
* **Linked service**: Select `ls_datalake` (the ADLS Gen2 linked service created earlier).
* **File path**: Specify the exact location where the file should be saved in your Data Lake:
  * **Container/Directory/File**: `bronze/onprem/firstcopy.csv`

**4. Validate and Execute**
* **Validate**: Click the **Validate** button at the top of the pipeline canvas. This checks your pipeline configuration for any missing settings or logical errors before running.
* **Debug**: Click the **Debug** button to manually trigger a test run of your pipeline. 
  * You can monitor the progress in the **Output** tab at the bottom.
  * ✅ **Result**: Once the debug run succeeds, your on-premises CSV file has been successfully copied to your Azure Data Lake Storage Gen2 container!

## 🔁 Dynamic Ingestion: Parameterizing the Pipeline with a ForEach Loop

### Why use a ForEach Loop and Parameters?
Instead of creating a separate pipeline or Copy Activity for every single file, we can make our pipeline **dynamic**. By using parameters and a `ForEach` loop, we can pass a list of filenames to a single Copy Activity, allowing it to iterate through the list and copy all the files automatically.

### ➕ Steps to Configure Dynamic Ingestion

#### 1. Create a Pipeline-Level Parameter
First, we need to define the list of files we want to iterate over at the pipeline level.
* Click on the blank canvas of your `onprem_ingestion` pipeline.
* Go to the **Parameters** tab at the bottom.
* Click **+ New** to create a parameter:
  * **Name**: `files`
  * **Type**: `Array`
  * **Value**: Paste the following JSON array representing our files:
    ```json
    [
      { "file_name": "Dimpassenger.csv" },
      { "file_name": "DimAirline.csv" },
      { "file_name": "DimFlight.csv" }
    ]
    ```

#### 2. Add and Configure the ForEach Activity
* From the **Activities** pane under **Iteration & conditionals**, drag the **ForEach** activity onto the canvas.
* Go to the **Settings** tab of the ForEach activity:
  * **Batch count**: Set to `50` (This controls how many files can be copied concurrently).
  * **Items**: Click inside the box, select **Add dynamic content**, and select your pipeline parameter (it will look like `@pipeline().parameters.files`).

#### 3. Parameterize the Source Dataset
We need a new source dataset that can accept a filename dynamically instead of having a hardcoded name.
* Go to the **Copy Activity** > **Source** tab > click **+ New**.
* Select **File System** > **CSV**.
* **Name**: `ds_onpre_src_param`
* Go to the dataset's **Advanced** / **Parameters** tab and click **+ New**:
  * **Name**: `p_file_name`
* Go back to the dataset's **Connection** tab. Next to the **File name** field, click **Add dynamic content** and select the parameter you just created: `@dataset().p_file_name`.

#### 4. Parameterize the Sink Dataset
We need to do the same for the destination dataset.
* Go to the **Copy Activity** > **Sink** tab and open your sink dataset.
* Remove any hardcoded filename in the connection tab.
* Go to the **Parameters** tab and click **+ New**:
  * **Name**: `p_filename`
* Go back to the **Connection** tab. Next to the **File name** field, click **Add dynamic content** and select: `@dataset().p_filename`.

#### 5. Move Copy Activity Inside ForEach & Map Values
Now we put the logic together inside the loop.
* Select your **Copy Activity** on the main canvas and press **Ctrl+X** (Cut).
* Double-click on the **ForEach** activity to open its inner canvas, and press **Ctrl+V** (Paste).
* Click on the Copy Activity inside the loop.
* **Map Source Parameter**: Go to the **Source** tab. In the dataset properties for `p_file_name`, click the value box, select **Add dynamic content**, and enter: `@item().file_name`
* **Map Sink Parameter**: Go to the **Sink** tab. In the dataset properties for `p_filename`, click the value box, select **Add dynamic content**, and enter: `@item().file_name`
  *(Note: `@item()` represents the current JSON object in the loop).*

#### 6. Execute the Pipeline
* Click **Validate** to ensure there are no configuration errors.
* Click **Debug** to run the pipeline.
* ✅ **Result**: The ForEach activity will loop 3 times, passing each filename dynamically to the Copy Activity. All three CSV files will be successfully migrated from your on-premises folder to your Azure Data Lake!